1. Import Libraries

In [4]:
import warnings
warnings.filterwarnings("ignore")
import os
import sys
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
sys.path.append(os.path.abspath(".."))
from src.preprocessing import preprocess
from src.feature_engineering import create_features


2. Load Train/Test Data


In [6]:
TRAIN_PATH = "../data/train.csv"
TEST_PATH  = "../data/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(train.shape)
print(test.shape)

train.head()


(690088, 15)
(295753, 14)


,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [8]:
# SAVE IDs
train_ids = train.id
test_ids = test.id


3. Preprocessing


In [9]:
train = preprocess(train)
test = preprocess(test)

print(train.shape)
print(test.shape)

(690088, 14)
(295753, 13)



4. New Feature Engineering (updated feature_engineering.py)


In [10]:
train = create_features(train)
test = create_features(test)

print(train.shape)
print(test.shape)

(690088, 34)
(295753, 33)


In [12]:
# varify new feature 
print("=" * 60)
print("Train Shape :", train.shape)
print("Test Shape  :", test.shape)

print("=" * 60)

train.head()

Train Shape : (690088, 34)
Test Shape  : (295753, 33)


,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,...,bmi_water,hydration_score,health_index,bmi_category,bmi_risk,stress_sleep,activity_diet,gender_activity,smoking_stress,diet_smoking
0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,...,47.7276,0.000855,26.88,overweight,over,high_average,sedentary_veg,female_sedentary,yes_high,veg_yes
1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,...,32.5584,0.000641,56.69,overweight,over,low_average,moderate_non-veg,other_moderate,yes_low,non-veg_yes
2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,...,39.2640,0.000595,44.99,normal,normal,high_poor,active_veg,male_active,yes_high,veg_yes
3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,...,46.7226,0.000768,66.62,normal,normal,high_average,active_veg,female_active,occasional_high,veg_occasional
4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,Unknown,...,63.9900,0.000879,55.48,overweight,over,Unknown_average,sedentary_veg,male_sedentary,Unknown_Unknown,veg_Unknown


In [13]:
# Prepare Data
TARGET = "health_condition"

X = train.drop(columns=[TARGET])

y = train[TARGET]

cat_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Number of Features :", X.shape[1])

print("Categorical Features :", len(cat_features))

Number of Features : 33
Categorical Features : 13



5. Train CatBoost
   (same best parameters)


In [14]:
train_pool = Pool(
    X,
    y,
    cat_features=cat_features
)

print("Pool created successfully.")

Pool created successfully.


In [18]:
model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    auto_class_weights="Balanced",
    random_seed=42,

    # GPU
    task_type="GPU",
    devices="0",

    # Training
    early_stopping_rounds=200,
    verbose=200
)
model

CatBoostClassifier(auto_class_weights='Balanced', depth=8, devices='0', early_stopping_rounds=200, eval_metric='TotalF1', iterations=3000, learning_rate=0.03, loss_function='MultiClass', random_seed=42, task_type='GPU', verbose=200)

In [19]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from catboost import Pool
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):

    print("=" * 60)
    print(f"Fold {fold}")

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    train_pool = Pool(
        X_train,
        y_train,
        cat_features=cat_features
    )

    valid_pool = Pool(
        X_valid,
        y_valid,
        cat_features=cat_features
    )

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True
    )

    pred = model.predict(valid_pool)

    score = balanced_accuracy_score(y_valid, pred)

    print(f"Balanced Accuracy : {score:.6f}")

    scores.append(score)

print("=" * 60)
print("CV Scores:", scores)
print("Mean CV :", np.mean(scores))
print("Std :", np.std(scores))

Fold 1
0:	learn: 0.8014100	test: 0.8014320	best: 0.8014320 (0)	total: 55.3ms	remaining: 2m 45s
200:	learn: 0.9490615	test: 0.9493508	best: 0.9493591 (194)	total: 6.87s	remaining: 1m 35s
400:	learn: 0.9502272	test: 0.9493599	best: 0.9495286 (289)	total: 13.7s	remaining: 1m 28s
bestTest = 0.9495286499
bestIteration = 289
Shrink model to first 290 iterations.
Balanced Accuracy : 0.949490
Fold 2
0:	learn: 0.8035045	test: 0.8054846	best: 0.8054846 (0)	total: 45.9ms	remaining: 2m 17s
200:	learn: 0.9486694	test: 0.9505584	best: 0.9505584 (200)	total: 7.01s	remaining: 1m 37s
400:	learn: 0.9497654	test: 0.9509312	best: 0.9509844 (392)	total: 13.6s	remaining: 1m 28s
600:	learn: 0.9505558	test: 0.9510284	best: 0.9511155 (565)	total: 20.2s	remaining: 1m 20s
800:	learn: 0.9511720	test: 0.9509746	best: 0.9511382 (750)	total: 26.9s	remaining: 1m 13s
bestTest = 0.951138188
bestIteration = 750
Shrink model to first 751 iterations.
Balanced Accuracy : 0.951095
Fold 3
0:	learn: 0.9206299	test: 0.9200397	


6. Cross Validation



7. Feature Importance



8. Generate Submission



9. Save Model